# RL Trading Victim V1 — Google Colab GitHub Runner

This notebook clones **`RL-trading-agent-robustness-evaluation/rltrade`** directly from GitHub and executes high-performance PPO agent training on Google Colab GPU / High-RAM instances.

## 1. System & GPU Check

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 2. Clone Repository from GitHub

> **Authentication for Private Repositories**:
> 1. In Colab, click the **Key icon (Secrets)** on the left sidebar 🔑.
> 2. Add a new secret with Name `GITHUB_TOKEN` and your GitHub Personal Access Token (classic with `repo` scope).
> 3. Enable "Notebook access" for this secret.
> 4. Run the cell below.

In [ ]:
import os

repo_owner = "RL-trading-agent-robustness-evaluation"
repo_name = "rltrade"

try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
    clone_url = f"https://{gh_token}@github.com/{repo_owner}/{repo_name}.git"
    print("Using GitHub token from Colab Secrets.")
except Exception:
    # Fallback to public clone if token is not set
    clone_url = f"https://github.com/{repo_owner}/{repo_name}.git"
    print("Attempting clone without token (public repository).")

!git clone {clone_url}
%cd {repo_name}

## 3. Install Package & Run Verification Tests (Gate L1 & Gate E2)

In [ ]:
!pip install -e .
!pip install tensorboard

# Run deterministic unit tests
!python -m pytest tests/test_ledger.py tests/test_env.py

## 4. Launch PPO Training on Canonical CRSP Dataset
Train across predeclared seeds per Regulation V1 §20: Seed 42, Seed 100, Seed 2026.

In [ ]:
# Train Primary Victim (Seed 42, 100,000 steps)
!python src/agent/train_ppo.py --train-csv data/processed/crsp_spy/train.csv --val-csv data/processed/crsp_spy/val.csv --timesteps 100000 --seed 42

In [ ]:
# Multi-seed replication runs
!python src/agent/train_ppo.py --train-csv data/processed/crsp_spy/train.csv --val-csv data/processed/crsp_spy/val.csv --timesteps 100000 --seed 100
!python src/agent/train_ppo.py --train-csv data/processed/crsp_spy/train.csv --val-csv data/processed/crsp_spy/val.csv --timesteps 100000 --seed 2026

## 5. Evaluate Financial Metrics & State-Responsiveness (Gate V1)

In [ ]:
!python src/agent/evaluate.py --model models/ppo_victim_v1_seed42.zip --data data/processed/crsp_spy/val.csv --output-ledger experiments/ppo_runs/crsp_val_ledger_seed42.csv

## 6. Live Interactive TensorBoard Inside Colab

In [ ]:
%load_ext tensorboard
%tensorboard --logdir experiments/ppo_runs/tensorboard

## 7. Save & Export Results
Choose **Option A** to commit & push weights/ledgers back to GitHub, or **Option B** to download a `.zip` directly to your computer.

In [ ]:
# Option A: Push trained models and step ledgers back to GitHub
# !git config --global user.name "Your Name"
# !git config --global user.email "your@email.com"
# !git add models/ experiments/
# !git commit -m "Add trained PPO victim v1 weights and logs from Colab"
# !git push origin master

In [ ]:
# Option B: Download results as a zip archive
from google.colab import files
!zip -r ppo_crsp_results.zip models/ experiments/
files.download('ppo_crsp_results.zip')